# Spectral animations with `animate_spectra` (archived)

Heterodyne, homodyne and $x_2$ position spectra of a 3×1 system, animated two ways with
`levisim.multiphysics.utils.animations.animate_spectra`: *slices* (spectra at each power-series
evaluation order, one frame per coupling $g$) and *heatmaps* (spectrum vs. $g$, one frame per
evaluation order).

**Runtime:** about a minute with `MAKE_ANIMATIONS = False` (coarse 4 kHz grid, 4 couplings, static
previews); with `MAKE_ANIMATIONS = True` (1 kHz grid, 50 couplings, six animations written to
`output/`) expect tens of minutes.

**Updated Nov 2026** to current levisim:

- `SpectrumType.PSD` → `SpectrumType.POSITION_PSD`; the toy parameter `'φ2'` → `'φ_even_n'`; animations are written to `output/` by
  `save_animation` instead of `save_mp4=True` (which needs ffmpeg) and are not embedded;
- the power series is generated (`generate_power_series=True`) because the 3×1 pickle is not tracked;
- the old `'pairs'` keys `first_Ωx`/`Ω_gap` are written as `Ω_first`, `Ω_interpair_gap` and
  `Ω_intrapair_gap = Ω_gap/10` (the original notebook has no stored output to compare with; in
  levisim versions from March 2025 on, those two old keys were silently ignored, giving Ω_first = 300 kHz);
- a workaround cell adapts `levisim.multiphysics.utils.animations`, which is out of date with
  `MultiTrap` (see the comment in that cell).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.animation import writers

from levisim.core.plotting import set_plt_rcParams_to_xmgrace_style

# MAKE_ANIMATIONS = True computes every frame at full resolution and writes the animations to
# output/ (mp4 with ffmpeg, else gif). False (the smoke-test setting) draws a quick static preview
MAKE_ANIMATIONS = False
OUTPUT_DIR = Path('output')

set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0


def save_animation(animation, name):
    """Writes `animation` to output/<name>.mp4 (or .gif without ffmpeg), never into the notebook"""
    OUTPUT_DIR.mkdir(exist_ok=True)
    writer = 'ffmpeg' if writers.is_available('ffmpeg') else 'pillow'
    path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
    animation.save(path, writer=writer)
    print(f'Wrote {path}')
    return path

In [ ]:
# WORKAROUND (Nov 2026): levisim.multiphysics.utils.animations still calls the pre-rename
# MultiTrap.calculate_*_spectrum methods, reads a MultiTrap.QLT_method attribute that MultiTrap no
# longer sets, sets the old toy parameter 'φ2' (now 'φ_even_n'), and uses np.trapz (removed in NumPy 2.4). Until that module is fixed, hand it a
# MultiTrap subclass that provides the old names. Delete this cell once animations.py is updated.
import numpy as np

import levisim.multiphysics.utils.animations as animations_module
from levisim.multiphysics.multitrap import MultiTrap
from levisim.multiphysics.enums.quantum_linear_theory_method import QuantumLinearTheoryMethod


class AnimationMultiTrap(MultiTrap):
    """MultiTrap with the method names levisim.multiphysics.utils.animations expects"""
    def __init__(self, **kwargs):
        # the phase of the 2nd tweezer is 'φ_even_n' since Dec 2025; animations.py still sets 'φ2'
        toy_params = kwargs.get('toy_params')
        if toy_params and 'φ2' in toy_params:
            kwargs['toy_params'] = {**toy_params, 'φ_even_n': toy_params['φ2']}
        super().__init__(**kwargs)
        self.QLT_method = kwargs.get('QLT_method', QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION)

    def calculate_heterodyne_spectrum(self, method):
        return self.heterodyne_spectrum(method)

    def calculate_homodyne_spectrum(self, method):
        return self.homodyne_spectrum(method)

    def calculate_all_position_power_spectra(self, method):
        # animations.py integrates these PSDs in m²/Hz, i.e. not zero-point scaled
        return self.all_position_power_spectra(method, zpf_scaled=False)


animations_module.MultiTrap = AnimationMultiTrap
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid

In [ ]:
import numpy as np
import matplotlib
from levisim import *

num_traps, Cartesian_axes_per_trap = 4, 1
num_traps, Cartesian_axes_per_trap = 5, 1
num_traps, Cartesian_axes_per_trap = 2, 2
num_traps, Cartesian_axes_per_trap = 3, 1

ω_range = np.arange(-400, 400, 1 if MAKE_ANIMATIONS else 4) * kHz

cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    # 'L': 0.96*cm,
    # 'Finesse': 40_000,
    'θhom': π/4,
}
gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}

default_g = -20*kHz
toy_params = {
    # the pre-2025 'pairs' keys first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    # 'Ω_model': 'equal spacing',
    # 'Ω_mid': 300*kHz,
    # 'Ω_span': 100*kHz,

    # 'Ω_model': 'custom',
    # 'toy_Ωjn_dict': {'x1': 300*kHz, 'y1': 301*kHz, 'x2': 302*kHz, 'y2': 303*kHz},

    'toy_gx': default_g,
    'toy_gy': default_g,

    # phase of the even-numbered tweezers (called 'φ2' before Dec 2025)
    'φ_even_n': 0*degrees,

    'use_Γ_formula': True,
}

multitrap_kwargs = {
    'num_traps': num_traps,
    'Cartesian_axes_per_trap': Cartesian_axes_per_trap,
    'ω_range': ω_range,

    'cavity_kwargs': cavity_kwargs,
    'gas_kwargs': gas_kwargs,
    'toy_params': toy_params,

    # only the 1×2, 2×2 and 3×2 power-series pickles are tracked in git, so generate it (instant for 3D)
    'generate_power_series': True,
    # 'max_order': 3,
}

multitrap = MultiTrap(**multitrap_kwargs)

# optical_damping.default_value = multitrap.cavity.κ
evaluation_order.default_value = multitrap.num_dimensions
# always evaluate the highest order (do this with NUMERICAL_HYBRIDISATION for speed)
evaluation_order.axis_range = np.array(
        list({multitrap.num_dimensions}.union({_ for _ in range(1, multitrap.max_order + 1)}))
    )
evaluation_order.range = evaluation_order.axis_range * evaluation_order.units
print(f'{evaluation_order.range=}')

light_mech_coupling.axis_range = np.linspace(-2, -100, 50 if MAKE_ANIMATIONS else 4)
light_mech_coupling.range = light_mech_coupling.axis_range * kHz

spectral_animations = {}
for spectrum_type in [
    SpectrumType.POSITION_PSD,
    SpectrumType.HETERODYNE,
    SpectrumType.HOMODYNE,
]:
    spectral_animations[spectrum_type] = {}
    for animation_type in [
        'slices',
        'heatmap',
    ]:
        if animation_type == 'slices':
            y, frame = evaluation_order, light_mech_coupling
        elif animation_type == 'heatmap':
            y, frame = light_mech_coupling, evaluation_order

        animation = animate_spectra(
            animation_type=animation_type,
            multitrap_kwargs=multitrap_kwargs,
            all_parameters=all_parameters,

            x=ω_range / kHz,
            y=y,
            frame=frame,

            spectrum_type=spectrum_type,
            PSD_label='x2',
            reverse_animation=animation_type == 'heatmap',
            # written below with save_animation, into output/
            save_mp4=False,
        )
        spectral_animations[spectrum_type][animation_type] = animation
        if MAKE_ANIMATIONS:
            save_animation(animation, f'{multitrap.num_traps}x{multitrap.Cartesian_axes_per_trap} {spectrum_type.name} {animation_type}')
